# Second Term Project: CQADupStack Collection

The *CQADupStack* is "[a] Benchmark Data Set for Community Question-Answering Research" [1] that is a part of the [*Benchmarking Information Retrieval (BEIR)*](https://github.com/beir-cellar/beir) collection.

CQADupStack contains data from 12 different [*Stackexchange*](https://stackexchange.com/) subforums based on the data dump released on September 26, 2014.

Your tasks, reviewed by your colleagues and the course instructors, are the following:



1. *Implement a ranked retrieval system*, [1, Chapter 6] which will produce a list of documents from the CQADupStack collection in a descending order of relevance to a query from the CQADupStack collection.
2. *Document your code* in accordance with [PEP 257](https://www.python.org/dev/peps/pep-0257/), ideally using [the NumPy style guide](https://numpydoc.readthedocs.io/en/latest/format.html#docstring-standard) as seen in the code from exercises.
   *Stick to a consistent coding style* in accordance with [PEP 8](https://www.python.org/dev/peps/pep-0008/).
3. *Reach at least 25% mean average precision at 10* [1, Section 8.4] with your system on the CQADupStack collection.
4. _[Upload an .ipynb file](https://is.muni.cz/help/komunikace/spravcesouboru#k_ss_1) with this Jupyter notebook to the homework vault in IS MU._ You MAY also include a brief description of your information retrieval system and a link to an external service such as [Google Colaboratory](https://colab.research.google.com/), [DeepNote](https://deepnote.com/), or [JupyterHub](https://iirhub.cloud.e-infra.cz/).







[1] Hoogeveen, Doris and Verspoor, Karin M. and Baldwin, Timothy. [*CQADupStack: A Benchmark Data Set for Community Question-Answering Research*](https://dl.acm.org/doi/10.1145/2838931.2838934). ACM, 2015.

## Import the utility tools from the git repository.

First, we will install [our library](https://github.com/MIR-MU/pv211-utils).

It may be necessary to restart the runtime to get the installed packages to work.

In [1]:
%%capture
!pip install --upgrade git+https://github.com/MIR-MU/pv211-utils.git@main

In [2]:
# check that you have the newer version of pv211-utils (0.2.8)
!pip show pv211-utils

Name: pv211_utils
Version: 2.0.8
Summary: Utilities for PV211 project
Home-page: https://gitlab.fi.muni.cz
Author: Michal Stefanik
Author-email: stefanik.m@fi.muni.cz
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: beir, faiss-cpu, gensim, google, gspread, ijson, ipython, ipywidgets, nltk, oauth2client, rank-bm25, scikit-learn, sympy, torch, tqdm
Required-by: 


## Define the necessary classes

These will eventually represent the Queries, Documents and Relevance Judgements from the CQADupStack collection.

Query and Document consist only of their IDs and bodies.
Judgements are also just a Set of Tuples that represent pairs of relevant Document-Query combinations.

In [3]:
from pv211_utils.beir.entities import BeirDocumentBase, BeirQueryBase, BeirJudgementBase
from typing import Set


class Query(BeirQueryBase):
    """
    A processed query form the Beir collection.

    Parameters
    ----------
    query_id : int
        The number
    body : str
        Text of a query

    """

    def __init__(self, query_id: int, body: str):
        super().__init__(query_id, body)

    def __str__(self):
        return self.body


class Document(BeirDocumentBase):
    """
    A processed document form the Beir collection.

    Parameters
    ----------
    document_id : str
        A unique identifier of the document.
    body : str
        The text of the document.

    """

    def __init__(self, document_id: str, body: str):
        super().__init__(document_id, body)

    def __str__(self):
        return self.body


BeirJudgements = Set[BeirJudgementBase]


## Loading the datasets
### CQADupStack contains 12 datasets that will be loaded and merged:
- Android
- English
- Gaming
- GIS
- Mathematica
- Physics
- Programmers
- Stats
- TeX
- Unix
- Webmasters
- WordPress

For more details: <a href=http://nlp.cis.unimelb.edu.au/resources/cqadupstack/>CQADupStack site</a>.



In [4]:
from pv211_utils.datasets import CQADupStackDataset

data = CQADupStackDataset()

/usr/local/lib/python3.11/dist-packages/pv211_utils/arqmath/loader.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.11/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/22998 [00:00<?, ?it/s]

  0%|          | 0/40221 [00:00<?, ?it/s]

  0%|          | 0/45301 [00:00<?, ?it/s]

  0%|          | 0/37637 [00:00<?, ?it/s]

...

  0%|          | 0/42269 [00:00<?, ?it/s]

  0%|          | 0/68184 [00:00<?, ?it/s]

  0%|          | 0/47382 [00:00<?, ?it/s]

  0%|          | 0/17405 [00:00<?, ?it/s]

  0%|          | 0/48605 [00:00<?, ?it/s]

In [5]:
documents = data.load_documents(document_class=Document)

train_queries = data.load_train_queries(query_class=Query)
train_judgements = data.load_train_judgements()

validation_queries = data.load_validation_queries(query_class=Query)
validation_judgements = data.load_validation_judgements()

## Implementation of information retrieval system

Here we will define our IR system. If you want to use your own class it must define a method name `search` that takes a query and returns documents in descending order of relevance to the query.

This example returns documents in a decreasing order according to
a [*Okapi BestMatch25+*](https://en.wikipedia.org/wiki/Okapi_BM25#Modifications) similarity score between the documents and the given query.

If you wish you might use [preprocessing](https://github.com/MIR-MU/pv211-utils/tree/main/pv211_utils/preprocessing) or [ensemble](https://github.com/MIR-MU/pv211-utils/blob/developer/pv211_utils/ensembles.py) techniques from our library.

In [ ]:
from sentence_transformers import SentenceTransformer
from pv211_utils.datasets import CQADupStackDataset
from pv211_utils.beir.eval import BeirEvaluation
from pv211_utils.beir.leaderboard import BeirLeaderboard
import torch


data = CQADupStackDataset(download_location='./data')
documents = data.load_documents()
test_queries = data.load_test_queries()
test_judgements = data.load_test_judgements()


class MyRetriever:
    def __init__(self, documents):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer('all-MiniLM-L6-v2', device=self.device)

        self.doc_ids = list(documents.keys())
        doc_texts = [doc.body for doc in documents.values()]

        print("calculating...")

        self.doc_embeddings = self.model.encode(
            doc_texts,
            batch_size=128,
            show_progress_bar=False,
            convert_to_tensor=True
        )
        self.id_to_obj = documents

    def search(self, query):
        query_emb = self.model.encode([query.body], convert_to_tensor=True, device=self.device)
        cos_scores = torch.nn.functional.cosine_similarity(query_emb, self.doc_embeddings)
        scores, indices = torch.topk(cos_scores, k=10)
        return [self.id_to_obj[self.doc_ids[i]] for i in indices.cpu().numpy()]

system = MyRetriever(documents)

## Evaluate the system on a given dataset

We will evaluate the IR system using the [Mean Average Precision](https://en.wikipedia.org/wiki/Evaluation_measures_(information_retrieval)#Mean_average_precision) (MAP).

In [ ]:
evaluation = BeirEvaluation(
    system,
    test_judgements,
    k=10,
    leaderboard=BeirLeaderboard(),
    author_name='Javeed, Umme Hani'
)

evaluation.evaluate(test_queries, submit_result=True)